# Mapping in Python

18 November 2026 · Lecture 3

Run Python cells from the repository root. Prepare local data with `uv run python scripts/fetch_data.py`. Paths are relative to that root.

## About the data

`countries.geojson` contains Natural Earth's simplified country boundaries (1:110 million, version 5.1.2), with a country name and ISO code per mapped area. Antarctica is omitted. Small territories may be absent and boundaries reflect the source's cartographic conventions, not a position on sovereignty.

`co2_2023.csv` contains 2023 country emissions estimates from Our World in Data and the Global Carbon Budget. Each row is a country that can be matched to our map. It includes population, territorial fossil/industrial CO₂ in million tonnes and tonnes per person, plus consumption-based estimates where available. Territorial emissions exclude land-use change. Missing values remain missing.

Sources, licences, selection rules and limitations are in the [data documentation](../../data/README.md). Files are prepared by `scripts/fetch_data.py`; this notebook reads local data only.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

# read_file loads the country polygons and their coordinate-system information.
countries = gpd.read_file("data/raw/countries.geojson")
# read_csv loads the emissions table; it contains values rather than shapes.
emissions = pd.read_csv("data/raw/co2_2023.csv")

## Geographical joins

Which countries emit the most CO₂ in total, and which emit most per person? These are
different questions. Join by ISO code, not by the spelling of country names. Retain map
areas without estimates and inspect them. The simplified map omits some small territories.

In [ ]:
# crs tells us which coordinate reference system the map uses.
print(countries.crs)
# on names the matching column; how="left" retains every country polygon.
# validate checks that each key occurs once in each table.
# indicator adds _merge, showing whether a row found a match.
world = countries.merge(emissions, on="iso_code", how="left", validate="one_to_one", indicator=True)
print(world[["iso_code", "map_name", "co2_per_capita", "_merge"]].head())

In [ ]:
# Inspect map areas that have no matching row in the emissions table.
unmatched = world["_merge"] != "both"
unmatched_areas = world.loc[unmatched, ["iso_code", "map_name"]]
print(unmatched_areas)
# isna() also finds missing values inside rows that did match a country.
missing_emissions = world["co2_per_capita"].isna()
missing_areas = world.loc[missing_emissions, ["map_name", "co2_per_capita"]]
print(missing_areas)

## Projections

Longitude and latitude are angles, not metres. A projection changes how Earth's surface
appears on the page. Equal Earth preserves relative areas and is useful for a world choropleth.
`to_crs` transforms coordinates; assigning a CRS label alone does not transform them.

In [ ]:
# Transform the shapes' coordinates to the Equal Earth projection.
# EPSG:8857 identifies this projection; it does not change the emissions.
projected = world.to_crs("EPSG:8857")
print(projected.crs)

In [ ]:
# Compare the original coordinates and the projection side by side.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# Give polygons a grey fill and thin white borders in each panel.
world.plot(ax=axes[0], color="lightgrey", edgecolor="white", linewidth=0.2)
projected.plot(ax=axes[1], color="lightgrey", edgecolor="white", linewidth=0.2)
axes[0].set_title("Longitude and latitude")
axes[1].set_title("Equal Earth")
# Hide coordinate ticks and frames to focus attention on the map shapes.
for ax in axes:
    ax.set_axis_off()
plt.show()

## Choropleth: emissions per person

Colour represents tonnes per person. Missing estimates are grey, not zero. Check the
unit conversion: the total is in million tonnes, while population counts people.

In [ ]:
# Check Austria and three large emitters against the supplied per-person values.
selected_codes = emissions["iso_code"].isin(["AUT", "USA", "CHN", "IND"])
check = emissions.loc[selected_codes].copy()
# Convert million tonnes to tonnes before dividing by the number of people.
total_tonnes = check["co2"] * 1_000_000
check["calculated_per_person"] = total_tonnes / check["population"]
print(check[["country", "co2", "co2_per_capita", "calculated_per_person"]])

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
# column supplies the values to colour; cmap selects yellow-to-red colours.
# legend=True adds a colour scale; legend_kwds supplies its label.
# missing_kwds gives missing values their own colour instead of treating them as 0.
projected.plot(column="co2_per_capita", cmap="YlOrRd", legend=True, ax=ax,
               legend_kwds={"label": "Territorial CO₂ (tonnes per person)"},
               missing_kwds={"color": "lightgrey", "label": "No estimate"})
ax.set_title("Territorial CO₂ emissions per person, 2023")
# Hide the coordinate axes so the map and colour scale take priority.
ax.set_axis_off()
# Keep the emissions definition, projection and missing-data note with the map.
fig.text(0.01, 0.01, "OWID / Global Carbon Budget; Natural Earth. Fossil fuels and industry; land-use change excluded.\nEqual Earth projection. Grey areas have no estimate; they are not zero.", fontsize=8)
# Export a PNG at 150 dots per inch, with margins fitted around the content.
fig.savefig("figures/03_co2_per_person.png", dpi=150, bbox_inches="tight")
plt.show()

## Proportional symbols: total emissions

Place one circle within each mapped country. These are cartographic positions, not
emission-source locations. Matplotlib's `s` controls area in points squared, so scaling
`s` with emissions makes area proportional to emissions. Include a size legend.

In [ ]:
# Only draw circles for reported totals; retain all map areas in the background.
has_total = projected["co2"].notna()
points = projected.loc[has_total].copy()
# Replace each polygon with one point inside it for placing its circle.
representative_points = points.geometry.representative_point()
print(representative_points.head())
points["geometry"] = representative_points

In [ ]:
# Use projected point coordinates and circle areas as separate plotting inputs.
point_x = points.geometry.x
point_y = points.geometry.y
circle_areas = points["co2"] / 10
fig, ax = plt.subplots(figsize=(11, 5.5))
# Draw the country polygons first as a pale background.
projected.plot(ax=ax, color="#eeeeee", edgecolor="white", linewidth=0.2)
# geometry.x and .y provide point coordinates in the projected map.
# s sets circle area in points squared; /10 keeps the symbols manageable.
# alpha=0.6 makes overlapping circles partly transparent.
ax.scatter(point_x, point_y, s=circle_areas, alpha=0.6, color="#176B73")
# Empty x/y lists add no map points, but create labelled symbols for the legend.
# Use the same size rule as above for a meaningful size legend.
for total in [100, 1000, 5000]:
    ax.scatter([], [], s=total / 10, color="#176B73", alpha=0.6, label=f"{total:,} million tonnes")
ax.legend(loc="lower left", fontsize=8)
ax.set_title("Total territorial CO₂ emissions, 2023")
ax.set_axis_off()
fig.text(0.01, 0.01, "OWID / Global Carbon Budget; Natural Earth. Circle area encodes total; locations are representative map points.", fontsize=8)
plt.show()

## Reading the maps fairly

Compare China, the US and India in the table and maps. Total emissions and emissions
per person rank countries differently. Neither measure is automatically the right one.
These territorial estimates count production within borders; consumption-based estimates
account for traded goods. Keep that distinction in the note.

Optional spatial-join check: recover each representative point's containing polygon.
This is a geometry check, not evidence about the location of factories or households.

In [ ]:
# A spatial join matches by location rather than by country code.
# predicate="within" finds the polygon containing each point.
# how="left" keeps all points, including any without a matching polygon.
point_locations = points[["iso_code", "geometry"]]
map_areas = projected[["map_name", "geometry"]]
located = gpd.sjoin(point_locations, map_areas, how="left", predicate="within")
print(located.head())

In [ ]:
# Missing names flag unmatched points; sum() counts the True results.
unmatched_points = located["map_name"].isna()
print("Unmatched points:", unmatched_points.sum())
# duplicated() flags each repeat after the first occurrence of a code.
repeated_codes = located["iso_code"].duplicated()
print("Repeated country codes:", repeated_codes.sum())

Open [the exercises](exercises.ipynb). Check join coverage, denominator, units, missing values and projection before keeping a map.

Course AI use and disclosure requirements are in [the policy](../../docs/ai_policy.md). If you use assistance, verify the code, numbers and interpretation and disclose it as required.